# Knowledge Sources

## String Knowledge Source

In [3]:
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource

# Define the knowledge
policy_text = """ Our return policy allows customers to return any product within 30 days of purchase.
    Refunds will be issued only if the item is unused and in original packaging.
    Customers must provide proof of purchase when requesting a return."""

# Create a String Knowledge Source Object
return_policy_knowledge = StringKnowledgeSource(content=policy_text)

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# CrewAI 1.15.1 adds a cache_breakpoint marker that Groq rejects.
# Disable that marker for this notebook so Groq requests stay compatible.
from crewai.llms import cache as crew_cache


def _patched_mark_cache_breakpoint(message):
    return message


crew_cache.mark_cache_breakpoint = _patched_mark_cache_breakpoint

In [5]:
from crewai import LLM

llm = LLM(model="groq/llama-3.3-70b-versatile")

In [6]:
from crewai import Agent

returns_agent = Agent(
    role = "Product Returns Assistant",
    goal = "Answer customer's question about return policy accurately.",
    backstory = "You work in customer service and specialize in returns, refunds, and policies.",
    allow_delegation=False,
    verbose=False,
    llm=llm
)

In [7]:
from crewai import Task

returns_task = Task(
    description = "Answer the following customer question about returns: {question}",
    expected_output = "A concise and accurate answer",
    agent = returns_agent
)

In [8]:
from crewai import Crew, Process

crew = Crew(
    agents = [returns_agent],
    tasks = [returns_task],
    process = Process.sequential,
    knowledge_sources = [return_policy_knowledge],
    verbose = True
)

In [14]:
result = await crew.kickoff_async(
    inputs={
        "question": "Can I get a refund if I used the item once ?"
    }
)

╭───────────────────────── 🚀 Crew Execution Started ──────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name: crew                                                                  │
│  ID: 78b327d4-d736-4f38-81f0-a00971dd6487                                    │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 📋 Task Started ───────────────────────────────╮
│                                                                              │
│  Task Started                                                                │
│  Name: Answer the following customer question about returns: Can I get a     │
│  refund if I used the item 

## Text Knowledge Source

In [19]:
from crewai.knowledge.source.text_file_knowledge_source import TextFileKnowledgeSource

text_source = TextFileKnowledgeSource(
    file_paths=["hr_policy.txt"]
)

In [20]:
from crewai import Agent, Task, Crew, Process, LLM

hr_agent = Agent(
    role = "HR Policy Assistent",
    goal = "Answer employee questions about HR Policies.",
    backstory = "You are a reliable HR knowledge assistant",
    knowledge_sources=[text_source],
    llm=llm
)

hr_task = Task(
    description="What is the leave policy for new employees ?",
    expected_output = "A clear summary of the leave policy",
    agent = hr_agent
)

In [21]:
crew = Crew(
    agents = [hr_agent],
    tasks = [hr_task],
    process = Process.sequential,
    verbose = True
)

result = await crew.kickoff_async()

╭───────────────────────── 🚀 Crew Execution Started ──────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name: crew                                                                  │
│  ID: 27231c7a-5b5f-477e-a421-646ab3ef8c15                                    │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 📋 Task Started ───────────────────────────────╮
│                                                                              │
│  Task Started                                                                │
│  Name: What is the leave policy for new employees ?                          │
│  ID: 2c21d1b3-6ebc-49fc-ac2

╭─────────────────────────────── Tracing Status ───────────────────────────────╮
│                                                                              │
│  Info: Tracing is disabled.                                                  │
│                                                                              │
│  To enable tracing, do any one of these:                                     │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯


## Pdf Source

In [22]:
from crewai.knowledge.source.pdf_knowledge_source import PDFKnowledgeSource

pdf_source = PDFKnowledgeSource(
    file_paths = ['meeting_notes.pdf']
)

In [23]:
meeting_summarizer = Agent(
    role = "Meeting Note Summarizer",
    goal = "Provide concise summaries of weekly meetings",
    backstory = "You help the team stay updated on discussions",
    knowledge_sources = [pdf_source]
)

In [24]:
meeting_summarizer = Agent( 
    role="Meeting Note Summarizer",
    goal="Provide concise summaries of weekly meetings.", 
    backstory="You help the team stay updated on discussions.", 
    knowledge_sources=[pdf_source],
    llm=llm
)

meeting_task = Task(
    description="Summarize the key action items from last week's meeting.",
    expected_output="A bullet-point list of action items.",
    agent=meeting_summarizer
)

In [25]:
crew = Crew(
    agents = [meeting_summarizer],
    tasks = [meeting_task],
    process = Process.sequential,
    verbose = True
)

result = await crew.kickoff_async()

╭───────────────────────── 🚀 Crew Execution Started ──────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name: crew                                                                  │
│  ID: ea14f99e-2a11-4cb5-9ae9-566432eda1ac                                    │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 📋 Task Started ───────────────────────────────╮
│                                                                              │
│  Task Started                                                                │
│  Name: Summarize the key action items from last week's meeting.              │
│  ID: 037ee19b-733d-4184-97d

In [26]:
from crewai.knowledge.source.csv_knowledge_source import CSVKnowledgeSource

csv_source = CSVKnowledgeSource(
    file_paths = ["feedback.csv"]
)

In [27]:
feedback_analyst = Agent(
    role = "User Feedback Analyst",
    goal = "Identity common themes in user feedback",
    backstory = "You specialize in converting raw feedback into insight",
    knowledge_sources = [csv_source],
    llm=llm
)

feedback_task = Task(
    description="What are the three most common complaints users had last month?",
    expected_output="A short list of recurring issues.",
    agent=feedback_analyst
)

In [29]:
crew = Crew(
    agents = [feedback_analyst],
    tasks = [feedback_task],
    process = Process.sequential,
    verbose = True
)

result = await crew.kickoff_async()

╭───────────────────────── 🚀 Crew Execution Started ──────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name: crew                                                                  │
│  ID: 933dd2f0-783f-439b-8cf4-3c7cb1d7c776                                    │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────── 🔍 Knowledge Retrieval ───────────────────────────╮
│                                                                              │
│  Knowledge Retrieval Started                                                 │
│  Status: Retrieving...                                                       │
│                            

/tmp/ipykernel_11028/2099782628.py:8: RuntimeWarning: coroutine 'Crew.kickoff_async' was never awaited
  result = await crew.kickoff_async()


## JSON source

In [31]:
from crewai.knowledge.source.json_knowledge_source import JSONKnowledgeSource

json_source = JSONKnowledgeSource(
    file_paths = ["company_info.json"]
)

In [32]:
company_expert = Agent( 
    role="Company Info Specialist", 
    goal="Answer questions about company structure and data.", 
    backstory="You are an internal data assistant for org-level queries.", 
    knowledge_sources=[json_source], 
    llm=llm
)

task = Task( 
    description="How many teams are working on the product and what are their names?", 
    expected_output="A list of team names and their sizes.",
    agent=company_expert
)

In [33]:
crew = Crew(
    agents=[company_expert],
    tasks=[task],
    process=Process.sequential,
    verbose=True,
    knowledge_sources=[json_source]
)

result = await crew.kickoff_async()

╭───────────────────────── 🚀 Crew Execution Started ──────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name: crew                                                                  │
│  ID: 5038763f-1ab4-44a4-afd4-f1ec61cafc6d                                    │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 📋 Task Started ───────────────────────────────╮
│                                                                              │
│  Task Started                                                                │
│  Name: How many teams are working on the product and what are their names?   │
│  ID: 6af0bf9f-8202-4f45-aa4

## Custom Embedding Model

In [44]:
from crewai import Memory

hf_embedder = Memory(embedder={
    "provider": "huggingface",
    "config": {
        "model_name": "sentence-transformers/all-MiniLM-L6-v2",
    },
})

In [45]:
from crewai. knowledge. source. string_knowledge_source import StringKnowledgeSource

# Internal onboarding FAQ
faq_content = """
    - You can access your email via portal.company.com using your employee credentials. - The standard work hours are from 9am to 6pm, Monday to Friday.
    - All reimbursement requests must be submitted by the 5th of the following month.
    - For any IT-related issues, contact support@company.com.
"""

# Create a string knowledge source 
faq_knowledge = StringKnowledgeSource(content=faq_content, embedder=hf_embedder)

In [46]:
from crewai import Agent

hr_faq_agent = Agent( 
    role="HR Assistant", 
    goal="Answer onboarding-related questions for new hires.",
    backstory="You are a helpful assistant who knows everything about internal policies and onboarding processes.", 
    allow_delegation=False,
    verbose=True, 
    embedder={
    "provider": "huggingface",
    "config": {
        "model_name": "sentence-transformers/all-MiniLM-L6-v2",
    },
}
)

In [47]:
from crewai import Task
task = Task( 
    description="Answer this onboarding question: {question}", 
    expected_output="A short, accurate answer based on internal HR documentation.", 
    agent=hr_faq_agent,
)

In [50]:
from crewai import Crew, Process
crew = Crew( 
    agents=[hr_faq_agent], 
    tasks=[task], 
    knowledge_sources=[faq_knowledge], 
    embedder={
        "provider": "huggingface",
        "config": {
            "model_name": "sentence-transformers/all-MiniLM-L6-v2",
        }, 
    },

    process=Process.sequential,
    verbose=True
)


result = await crew.kickoff_async(
    inputs={ 
        "question": "What are the working hours and how do I get reimbursed?"
    }
)

from pprint import pprint 

pprint(result. raw)


[2026-07-12 10:12:52][ERROR]: Failed to upsert documents: An embedding function already exists in the collection configuration, and a new one is provided. If this is intentional, please embed documents separately. Embedding function conflict: new: huggingface vs persisted: openai

[2026-07-12 10:12:52][WARNING]: Failed to init knowledge: An embedding function already exists in the collection configuration, and a new one is provided. If this is intentional, please embed documents separately. Embedding function conflict: new: huggingface vs persisted: openai
╭───────────────────────── 🚀 Crew Execution Started ──────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name: crew                                                                  │
│  ID: df2fbe7d-0b20-419a-a98a-658de182eb4a                                    │
│                               

ERROR:root:Error during knowledge search: An embedding function already exists in the collection configuration, and a new one is provided. If this is intentional, please embed documents separately. Embedding function conflict: new: huggingface vs persisted: openai
Traceback (most recent call last):
  File "/home/vibhav-ucharia/PyVenv/.venv/lib/python3.13/site-packages/crewai/knowledge/storage/knowledge_storage.py", line 78, in search
    return client.search(
           ~~~~~~~~~~~~~^
        collection_name=collection_name,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
        score_threshold=score_threshold,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/vibhav-ucharia/PyVenv/.venv/lib/python3.13/site-packages/crewai/rag/chromadb/client.py", line 449, in search
    collection = self.client.get_or_create_collection(
        name=_sanitize_collection_name(params.collection_name),
        embedding_function=self.embedding_function,
    )
  File "/home/

╭─────────────────────────── 📚 Knowledge Retrieved ───────────────────────────╮
│                                                                              │
│  Search Query:                                                               │
│  Onboarding question: What are the official working hours and the detailed   │
│  process for expense reimbursement according to internal HR documentation?   │
│  Knowledge Retrieved:                                                        │
│  No knowledge retrieved                                                      │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 🤖 Agent Started ──────────────────────────────╮
│                                                                              │
│  Agent: HR Assistant       

## Custom Knowledge Source

In [61]:
from crewai.knowledge.source.base_knowledge_source import BaseKnowledgeSource
from typing import Dict, Any 
from pydantic import Field

import requests 

class WeatherKnowledgeSource(BaseKnowledgeSource) :
    """Knowledge source that fetches weather data from an external API."""
    city: str = Field(description="City for which weather should be fetched")

    def load_content(self) -> Dict[Any, str]:
        try: 
            print(f"Fetching weather for {self.city}...")

        # Open-Meteo API (no key needed for basic data) 
            endpoint = "https://api.open-meteo.com/v1/forecast" 
            params = {
                "latitude": 37.77, # San Francisco by default
                "Longitude": -122.42,
                "current_weather": True
            }

            response = requests.get(endpoint, params=params)
            response.raise_for_status()
    

            weather_data = response.json().get("current_weather", {}) 
            formatted = self.validate_content(weather_data)
            return {self.city: formatted}

        except Exception as e: 
            raise ValueError(f"Failed to fetch weather data: {str(e)}")


    def validate_content(self, data: dict) -> str: 
        if not data: 
            return "No weather data available."

        return (
            f"Current weather in {self.city}:\n"
            f"- Temperature: {data.get('temperature')}°C\n"
            f"- Wind Speed: {data.get('windspeed')} km/h\n"
            f"- Weather Code: {data.get('weathercode')}\n"
            f"- Time: {data.get('time')}"
        )


    def add(self) -> None: 
        """Brocess and chunk the content.""" 
        
        content = self. load_content() 
        for _, text in content.items():
            chunks = self._chunk_text(text) 
            self.chunks.extend(chunks)
        self._save_documents()

    async def aadd(self) -> None:
        self.add()


In [62]:
from crewai import Agent, LLM 



weather_knowledge = WeatherKnowledgeSource(city="San Francisco")

weather_agent = Agent( 
    role="Weather Reporter", 
    goal="Answer questions about the current weather forecast.", 
    backstory="You are a friendly meteorologist who provides real-time weather updates.", 
    knowledge_sources=[weather_knowledge], 
    llm=llm, 
    verbose=True
)

In [63]:
from crewai import Task, Crew, Process 

task = Task( 
    description="What is the current temperature and wind speed in San Francisco?", 
    expected_output="A concise weather summary for San Francisco.",
    agent=weather_agent 
)

crew = Crew( 
    agents=[weather_agent], 
    tasks=[task], 
    process=Process.sequential,
    verbose=True
)

In [64]:
result = await crew.kickoff_async()

╭───────────────────────── 🚀 Crew Execution Started ──────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name: crew                                                                  │
│  ID: f40f71be-3e2c-4a31-90c5-5c3a6b881a54                                    │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

Fetching weather for San Francisco...
╭──────────────────────────────── Crew Failure ────────────────────────────────╮
│                                                                              │
│  Crew Execution Failed                                                       │
│  Name: crew                                                          

ValueError: Invalid Knowledge Configuration: Failed to fetch weather data: 400 Client Error: Bad Request for url: https://api.open-meteo.com/v1/forecast?latitude=37.77&Longitude=-122.42&current_weather=True